In [1]:
import os
os.chdir("..")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from optimizer import grid_search, strategy_compare

In [3]:
SEARCH_SPACE = {
    "threshold_ma200_sell": [1.5, 2.0],
    "cooldown_long": [30, 60, 90],
    "cooldown_short": [7, 14, 30],
    "minimum_profit": [1.1, 1.5, 2.0],
}

In [5]:
df_prices = pd.read_csv("data/test_crypto.csv", index_col=0, parse_dates=True)
df_prices

,BTC,ETH,SOL,XRP,DOGE,ADA,DOT,LINK,XLM,XMR
Date,,,,,,,,,,
2014-09-17,457.334015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-18,424.440002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-19,394.795990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-20,408.903992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-21,398.821014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2026-10-20,82941.000000,2446.730713,106.422340,1.334452,0.081809,0.223782,1.149771,12.320927,0.181960,600.929504
2026-10-21,84011.968750,2470.766113,106.793655,1.363619,0.081867,0.224979,1.141323,12.484719,0.183488,622.434082
2026-10-22,81311.890625,2386.189209,103.542320,1.307656,0.078676,0.212025,1.111869,11.943166,0.175209,600.638428


# SEARCH

In [ ]:
history_search = grid_search(
    prices=df_prices, 
    param_grid=SEARCH_SPACE, 
    n_boot=100,
    sample_size=10,
    sequence_length=365 * 4,
    )

 87%|████████▋ | 47/54 [48:22<07:28, 64.14s/it] 

In [ ]:
history_search

,strategy,Target,Cash_spent,Realized_cash,Portfolio,Invested,Total_value,Bull_history,Avg_price,Num_take_profits,MDD,MDD_usd,MA200_survival_days,RoAC,iteration
0,threshold_ma200_sell=1.5_threshold_bolinger_se...,DASH,1900,654,958,1306,1612,12,42.622593,8,-51,-410,53.833333,-25,1
1,threshold_ma200_sell=1.5_threshold_bolinger_se...,BNB,1544,809,1553,947,2362,78,546.097672,5,-25,-753,58.000000,53,1
2,threshold_ma200_sell=1.5_threshold_bolinger_se...,ONT,1900,1185,1089,1329,2274,47,0.318769,12,-46,-241,79.000000,123,1
3,threshold_ma200_sell=1.5_threshold_bolinger_se...,ADA,1603,2206,336,174,2542,19,0.486429,7,-43,-248,68.500000,52,1
4,threshold_ma200_sell=1.5_threshold_bolinger_se...,EOS,1851,852,606,1084,1459,6,0.849249,7,-47,-498,70.666667,-28,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647995,threshold_ma200_sell=2.0_threshold_bolinger_se...,BCH,1544,1583,1201,918,2784,69,264.414808,2,-26,-909,26.000000,122,100
647996,threshold_ma200_sell=2.0_threshold_bolinger_se...,MKR,1742,0,3712,1742,3712,40,1086.764372,4,-44,-2733,38.727273,19,100
647997,threshold_ma200_sell=2.0_threshold_bolinger_se...,NEO,1900,427,969,1702,1397,32,12.481918,10,-52,-673,136.000000,33,100
647998,threshold_ma200_sell=2.0_threshold_bolinger_se...,EOS,1782,0,409,1782,409,26,0.415311,4,-80,-1490,0.000000,-15,100


In [ ]:
search_res = history_search.groupby("strategy").agg(
    RoAC=("RoAC", "mean"),
    MDD=("MDD", "mean"),
    Portfolio=("Portfolio", "mean"),
    Realized_cash=("Realized_cash", "mean"),
    Cash_spent=("Cash_spent", "mean"),
    Total_value_mean=("Total_value", "mean"),
    Total_value_meadian=("Total_value", "median"),
    Total_value_low=("Total_value", lambda s: np.percentile(s, 25)),
    Total_value_high=("Total_value", lambda s: np.percentile(s, 75)),
)

search_res.sort_values("RoAC", ascending=False).style.background_gradient(cmap="RdYlGn", axis=0).format("{:.2f}")

,RoAC,MDD,MDD_usd,Portfolio,Realized_cash,Cash_spent,Total_value_mean,Total_value_meadian,Total_value_low,Total_value_high
strategy,,,,,,,,,,
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=70.0_threshold_ppo_sell=10.0_cooldown_long=60_cooldown_short=30_minimum_profit=2.0,108.62,-43.37,-1002.74,1296.47,1131.42,1686.13,2428.21,2126.50,1543.75,2842.00
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=70.0_threshold_ppo_sell=15.0_cooldown_long=30_cooldown_short=30_minimum_profit=1.1,107.46,-39.63,-598.07,881.89,1434.77,1671.92,2317.14,2201.50,1727.75,2638.00
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=70.0_threshold_ppo_sell=5.0_cooldown_long=60_cooldown_short=30_minimum_profit=2.0,107.43,-42.14,-948.85,1256.31,1161.32,1677.78,2417.97,2122.50,1549.00,2811.25
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.5_threshold_rsi_sell=70.0_threshold_ppo_sell=10.0_cooldown_long=30_cooldown_short=30_minimum_profit=1.1,106.37,-39.07,-576.95,856.08,1468.71,1682.94,2325.27,2160.00,1703.00,2662.50
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=70.0_threshold_ppo_sell=10.0_cooldown_long=30_cooldown_short=14_minimum_profit=2.0,106.06,-41.37,-778.08,1164.35,1160.04,1675.56,2324.73,2164.00,1548.75,2883.25
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.5_threshold_rsi_sell=70.0_threshold_ppo_sell=15.0_cooldown_long=30_cooldown_short=14_minimum_profit=1.5,103.31,-38.53,-562.62,982.95,1353.65,1678.37,2337.03,2182.00,1722.25,2614.25
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=70.0_threshold_ppo_sell=10.0_cooldown_long=30_cooldown_short=14_minimum_profit=1.1,103.23,-37.86,-486.07,798.12,1497.35,1662.60,2295.96,2149.00,1750.50,2603.75
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=70.0_threshold_ppo_sell=10.0_cooldown_long=30_cooldown_short=30_minimum_profit=1.5,102.58,-37.78,-567.75,973.81,1382.76,1676.53,2356.99,2211.50,1780.50,2658.25
threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=80.0_threshold_ppo_sell=5.0_cooldown_long=30_cooldown_short=30_minimum_profit=2.0,102.51,-40.32,-724.58,1148.97,1170.51,1678.03,2319.83,2129.50,1554.00,2847.25


In [ ]:
search_res.loc['threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=70.0_threshold_ppo_sell=10.0']

KeyError: 'threshold_ma200_sell=1.5_threshold_bolinger_sell=2.0_threshold_rsi_sell=70.0_threshold_ppo_sell=10.0'